# LeetCode #1191: K Concatenation Maximum Sum

https://leetcode.com/problems/k-concatenation-maximum-sum/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \times k)$ | $O(n \times k)$ |
| **Optimal: Kadane + Mathematical Insight ★** | $O(n)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Physically concatenate the array $k$ times, then run Kadane's algorithm on the resulting array of length $n \times k$. Correct but $O(nk)$ time and space, infeasible for large $k$.

### Optimal: Kadane + Mathematical Insight ★
Observe that the maximum subarray in $k$ copies can span at most two copies if the total sum $\leq 0$, or gain an extra `(k-2) * totalSum` from the interior copies if total sum $> 0$. Run Kadane on the double-copy `arr + arr` to capture the cross-boundary case, then add the interior contribution.

**Why this is better than Brute Force:** Reduces from $O(nk)$ to $O(n)$ by replacing physical concatenation with a mathematical multiplier.

**Constraints:**
* $1 \leq n \leq 10^5$
* $1 \leq k \leq 10^5$
* $-10^4 \leq \text{arr}[i] \leq 10^4$
* Result modulo $10^9 + 7$


## Solutions

### C#

In [ ]:
public class Solution {
    private const int MOD = 1_000_000_007;

    public int KConcatenationMaxSum(int[] arr, int k) {
        long total = 0;
        foreach (int x in arr) total += x;

        // Kadane on single copy (for k == 1 or total <= 0 cases)
        long single = Kadane(arr, 1);
        if (k == 1) return (int)(single % MOD);

        // Kadane on double copy captures the cross-boundary maximum subarray
        long doubleCopy = Kadane(arr, 2);

        // Interior full copies contribute only if the total sum is positive
        long extra = total > 0 ? (k - 2) * total : 0;

        return (int)((doubleCopy + extra) % MOD);
    }

    private long Kadane(int[] arr, int reps) {
        long maxSum = 0, cur = 0;
        for (int r = 0; r < reps; r++)
            foreach (int x in arr) {
                cur = Math.Max(cur + x, 0); // reset to 0 rather than go negative
                maxSum = Math.Max(maxSum, cur);
            }
        return maxSum;
    }
}

### Python

In [ ]:
class Solution:
    MOD = 10**9 + 7

    def kConcatenationMaxSum(self, arr: list[int], k: int) -> int:
        total = sum(arr)

        def kadane(reps: int) -> int:
            # Maximum subarray sum over reps copies; resets to 0 (never go negative)
            max_sum = cur = 0
            for _ in range(reps):
                for x in arr:
                    cur = max(cur + x, 0)
                    max_sum = max(max_sum, cur)
            return max_sum

        if k == 1:
            return kadane(1) % self.MOD

        # Double copy covers subarrays that cross the boundary between two copies
        double_max = kadane(2)
        # Interior copies each add a full totalSum if it is positive
        extra = max(k - 2, 0) * total if total > 0 else 0

        return (double_max + extra) % self.MOD

### Go

In [ ]:
func kConcatenationMaxSum(arr []int, k int) int {
    const MOD = 1_000_000_007

    total := 0
    for _, x := range arr {
        total += x
    }

    kadane := func(reps int) int {
        // Find the maximum contiguous subarray sum across reps copies
        maxSum, cur := 0, 0
        for r := 0; r < reps; r++ {
            for _, x := range arr {
                cur += x
                if cur < 0 {
                    cur = 0 // never let negative prefix drag the next candidate down
                }
                if cur > maxSum {
                    maxSum = cur
                }
            }
        }
        return maxSum
    }

    if k == 1 {
        return kadane(1) % MOD
    }

    doubleMax := kadane(2)
    extra := 0
    if total > 0 {
        extra = (k - 2) * total
    }
    return (doubleMax + extra) % MOD
}

### Rust

In [ ]:
impl Solution {
    pub fn k_concatenation_max_sum(arr: Vec<i32>, k: i32) -> i32 {
        const MOD: i64 = 1_000_000_007;

        let total: i64 = arr.iter().map(|&x| x as i64).sum();

        let kadane = |reps: usize| -> i64 {
            // Maximum subarray sum over reps concatenated copies
            let (mut max_sum, mut cur) = (0i64, 0i64);
            for _ in 0..reps {
                for &x in &arr {
                    cur = (cur + x as i64).max(0);
                    max_sum = max_sum.max(cur);
                }
            }
            max_sum
        };

        if k == 1 {
            return (kadane(1) % MOD) as i32;
        }

        let double_max = kadane(2);
        let extra = if total > 0 { (k as i64 - 2) * total } else { 0 };

        ((double_max + extra) % MOD) as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [1,2]`, `k = 3`
Total = 3 > 0. Kadane on `[1,2,1,2]` = 6. Extra = `(3-2) * 3 = 3`. Result = **9** (three full copies).

### 2. Slightly Complex
**Input:** `arr = [1,-2,1]`, `k = 5`
Total = 0 (not positive, no interior gain). Kadane on double copy `[1,-2,1,1,-2,1]`: best subarray is `[1]` or `[1,-2,1,1] = 1`? Max = 2 from `[1,-2,1,1]`? Let's trace: `cur: 1,0,1,2,0,1` → maxSum = **2**. No extra. Result = **2**.

### 3. Edge Case: Time Factor
**Input:** $n = 10^5$, $k = 10^5$, all `arr[i] = 1`
Total = $10^5 > 0$. Kadane runs on $2n = 2 \times 10^5$ elements. Extra = $(k-2) \times n$. All work is $O(n)$, with $O(1)$ multiplication for the extra term — confirming no $O(nk)$ blowup.

### 4. Edge Case: Space Factor
**Input:** $n = 10^5$, $k = 2$, all `arr[i] = -1`
Total = $-10^5 \leq 0$. Kadane returns 0 (no positive subarray). Extra = 0. Result = **0**. Only a handful of integers allocated — $O(1)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [-10000, 10001, -10000]`, `k = 2`
Total = $-9999 \leq 0$. Kadane on double copy: best is `[10001]` = 10001. No extra. Result = **10001**. Confirms that a single element can dominate even when the full-array sum is negative and $k > 1$.
